In [33]:
import pandas as pd


In [34]:
class DataLoader:
  def __init__(self, filepath, drop_cols=None):
    self.filepath = filepath
    self.drop_cols = drop_cols if drop_cols is not None else [
        "UDI", "Product ID", "TWF", "HDF","PWF","OSF","RNF"
    ]

    self.df = None #dataframe'i daha yüklemedik, boş başlatıyoruz

  def load(self):
    self.df = pd.read_csv(self.filepath)
    self.df = self.df.drop(self.drop_cols, axis=1)
    self.df = pd.get_dummies(self.df, columns=["Type"],drop_first=True)
    return self.df


In [35]:
loader = DataLoader("ai4i2020.csv")
df = loader.load()
df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,False,True
1,298.2,308.7,1408,46.3,3,0,True,False
2,298.1,308.5,1498,49.4,5,0,True,False
3,298.2,308.6,1433,39.5,7,0,True,False
4,298.2,308.7,1408,40.0,9,0,True,False


In [36]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

In [37]:
class Preprocessor():
  def __init__(self,df,random_state=42):
    self.df=df
    self.random_state=random_state
    self.scaler = StandardScaler()

  def prepare(self):
    #features ve target olarak ayırıyoruz
    x= self.df.drop("Machine failure", axis=1).values
    y= self.df["Machine failure"].values

    # %60 train , %20 validation , %20 test
    x_train, x_temp, y_train, y_temp = train_test_split(
        x,y, test_size=0.4, random_state=self.random_state, stratify=y
    )
    x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=self.random_state, stratify=y_temp
)
    #scaling sadece trainde fit vardır
    x_train = self.scaler.fit_transform(x_train)
    x_val = self.scaler.transform(x_val)
    x_test = self.scaler.transform(x_test)

    #oversample yine sadece trainde yapılır eşit olamayan sınıflarda az olanı çoğaltır
    ros = RandomOverSampler(random_state=self.random_state)
    x_train, y_train = ros.fit_resample(x_train, y_train)

    self.x_train, self.y_train = x_train, y_train
    self.x_val, self.y_val = x_val, y_val
    self.x_test, self.y_test = x_test, y_test

    return x_train, x_val, x_test, y_train, y_val, y_test

In [38]:
prep = Preprocessor(df)
x_train, x_val, x_test, y_train, y_val, y_test = prep.prepare()
print("Train dağılımı:", np.bincount(y_train))

Train dağılımı: [5797 5797]


In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [41]:
class BaseModel:
  def __init__(self, model=None):
    self.model = model

  def train(self, x_train, y_train):
    self.model.fit(x_train, y_train)

  def predict(self,x):
    return self.model.predict(x)

  def evaluate(self,x,y):
    y_pred = self.predict(x)
    print(classification_report(y,y_pred))
    print(confusion_matrix(y,y_pred))








In [42]:
class LogRegModel(BaseModel):
  def __init__(self):
    super().__init__(LogisticRegression())


In [43]:
log = LogRegModel()
log.train(x_train, y_train)
log.evaluate(x_val, y_val)


              precision    recall  f1-score   support

           0       0.99      0.84      0.91      1932
           1       0.15      0.82      0.26        68

    accuracy                           0.84      2000
   macro avg       0.57      0.83      0.58      2000
weighted avg       0.96      0.84      0.89      2000

[[1621  311]
 [  12   56]]


In [44]:
class KNNModel(BaseModel):
  def __init__(self, n_neighbors=5):
    super().__init__(KNeighborsClassifier(n_neighbors=n_neighbors))

In [45]:
knn = KNNModel(n_neighbors=5)
knn.train(x_train, y_train)
knn.evaluate(x_val, y_val)

              precision    recall  f1-score   support

           0       0.99      0.96      0.97      1932
           1       0.36      0.62      0.45        68

    accuracy                           0.95      2000
   macro avg       0.67      0.79      0.71      2000
weighted avg       0.96      0.95      0.96      2000

[[1856   76]
 [  26   42]]


In [46]:
import tensorflow as tf
tf.random.set_seed(42)

In [47]:
class NNModel(BaseModel):
  def __init__(self,input_dim):
     model =  tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')])
     model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
     super().__init__(model)

  def train(self, x_train, y_train, epochs=20, batch_size=32):
    self.model.fit(x_train,y_train, epochs=epochs, batch_size=batch_size, verbose=0)

  def predict(self,x):
    return (self.model.predict(x)>0.5).astype(int)










In [ ]:
nn = NNModel(input_dim=x_train.shape[1])
nn.train(x_train, y_train)
nn.evaluate(x_test, y_test)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
